# Criação das tabelas do ETL

Este notebook cria o schema MySQL usado pelo ETL de exercícios.

A operação usa `CREATE TABLE IF NOT EXISTS`, portanto não apaga dados nem recria tabelas existentes. Execute antes do ETL e configure `MYSQL_URL` sem gravar credenciais no notebook.


In [ ]:
import os

from sqlalchemy import create_engine, text

user='root'
password='*'
host='localhost'
port=3306
database='spring_api_system'

MYSQL_URL = f"mysql+pymysql://{user}:{password}@{host}:{port}/{database}"

if not MYSQL_URL:
    raise RuntimeError(
        'Defina a variável de ambiente MYSQL_URL antes de criar as tabelas.'
    )

engine = create_engine(MYSQL_URL)


In [11]:
def create_tables(confirm=False):
    """Cria as tabelas do ETL sem apagar tabelas ou dados existentes."""
    if confirm is not True:
        raise ValueError(
            'Operação cancelada. Execute create_tables(confirm=True) somente após confirmar.'
        )

    statements = [
        """
        CREATE TABLE IF NOT EXISTS body_part (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(100) NOT NULL UNIQUE
        ) ENGINE=InnoDB
        """,
        """
        CREATE TABLE IF NOT EXISTS equipment (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(100) NOT NULL UNIQUE
        ) ENGINE=InnoDB
        """,
        """
        CREATE TABLE IF NOT EXISTS secondary_muscle (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(100) NOT NULL UNIQUE
        ) ENGINE=InnoDB
        """,
        """
        CREATE TABLE IF NOT EXISTS exercise (
            id INT AUTO_INCREMENT PRIMARY KEY,
            name VARCHAR(255) NOT NULL,
            gif_url VARCHAR(255),
            instructions_sumarization TEXT,
            fk_body_part INT,
            fk_equipment INT,
            CONSTRAINT fk_exercise_body_part
                FOREIGN KEY (fk_body_part) REFERENCES body_part(id),
            CONSTRAINT fk_exercise_equipment
                FOREIGN KEY (fk_equipment) REFERENCES equipment(id)
        ) ENGINE=InnoDB
        """,
        """
        CREATE TABLE IF NOT EXISTS exercise_secondary_muscle (
            exercise_id INT NOT NULL,
            secondary_muscle_id INT NOT NULL,
            PRIMARY KEY (exercise_id, secondary_muscle_id),
            CONSTRAINT fk_esm_exercise
                FOREIGN KEY (exercise_id) REFERENCES exercise(id),
            CONSTRAINT fk_esm_secondary_muscle
                FOREIGN KEY (secondary_muscle_id) REFERENCES secondary_muscle(id)
        ) ENGINE=InnoDB
        """
    ]

    try:
        with engine.begin() as connection:
            for statement in statements:
                connection.execute(text(statement))
    except Exception:
        print('Falha ao criar as tabelas. A transação foi revertida.')
        raise

    print(
        'Tabelas criadas ou já existentes: body_part, equipment, '
        'secondary_muscle, exercise e exercise_secondary_muscle.'
    )


## Executar a criação

A chamada está comentada por segurança. Remova o comentário somente depois de confirmar o banco configurado:


In [12]:
create_tables(confirm=True)


Tabelas criadas ou já existentes: body_part, equipment, secondary_muscle, exercise e exercise_secondary_muscle.
